# 决策树第八课：CART 回归树

本课目标：补全 CART 的回归树部分，理解回归树如何用平方误差选择划分。

一句话版：**CART 分类树看 Gini，CART 回归树看平方误差。**

## 1. CART 回归树解决什么问题？

CART 的全称是 Classification And Regression Tree，也就是分类与回归树。

前面我们学的是 CART 分类树，用来预测类别：

- 是否生还：$0$ 或 $1$
- 是否批准贷款：批准或拒绝

CART 回归树用来预测连续值：

- 房价
- 收入
- 温度
- 销量

所以分类树和回归树的区别可以先记成：

$$\text{分类树预测类别，回归树预测数值。}$$

## 2. 分类树和回归树的划分标准不同

| 类型 | 预测目标 | 节点里关心什么 | 常用划分指标 |
| --- | --- | --- | --- |
| CART 分类树 | 类别 | 类别是否纯 | Gini 指数 |
| CART 回归树 | 连续值 | 数值是否集中 | 平方误差 SSE / MSE |

分类树希望切完以后，每个节点里的类别尽量单一。

回归树希望切完以后，每个节点里的数值尽量接近。

比如预测房价，某个叶子节点里的房价如果是：

$$\{120, 125, 123\}$$

这组很集中，适合作为一个叶子。

如果是：

$$\{80, 150, 300\}$$

这组差异很大，说明还应该继续划分。

## 3. 回归树的叶子节点预测什么？

CART 回归树的叶子节点通常预测该叶子中训练样本标签的平均值。

比如某个叶子节点里有 3 个房价：

$$\{100, 110, 120\}$$

这个叶子的预测值就是：

$$\hat{y}=\frac{100+110+120}{3}=110$$

为什么用平均值？因为在平方误差意义下，平均值能让误差最小。

$$SSE=\sum_{i=1}^{n}(y_i-\hat{y})^2$$

当 $\hat{y}$ 取样本均值时，$SSE$ 最小。

## 4. 回归树如何评价一个节点混不混？

回归树没有“类别纯度”，所以不用 Gini。

它看的是节点内数值离均值有多远，常用平方误差：

$$SSE(D)=\sum_{i=1}^{n}(y_i-\bar{y})^2$$

其中：

$$\bar{y}=\frac{1}{n}\sum_{i=1}^{n}y_i$$

$SSE$ 越小，说明节点里的数值越集中。

有时也会看均方误差：

$$MSE(D)=\frac{1}{n}\sum_{i=1}^{n}(y_i-\bar{y})^2$$

$MSE$ 就是把 $SSE$ 除以样本数。

## 5. 划分后的平方误差怎么算？

假设用某个条件把数据分成左右两个子节点：

$$D \rightarrow D_{left},D_{right}$$

那么这次划分后的总平方误差是：

$$SSE_{split}=SSE(D_{left})+SSE(D_{right})$$

CART 回归树选择：

$$\text{划分后 }SSE_{split}\text{ 最小的切法。}$$

这个思想和 CART 分类树很像：

- 分类树：选择划分后加权 Gini 最小的切法。
- 回归树：选择划分后平方误差最小的切法。

## 6. 手算例子：用面积预测房价

假设我们有 5 套房子：

| 编号 | 面积 | 房价 |
| ---: | ---: | ---: |
| 1 | 50 | 100 |
| 2 | 60 | 120 |
| 3 | 80 | 160 |
| 4 | 100 | 200 |
| 5 | 120 | 240 |

现在用 `面积` 这个连续特征做划分。

候选阈值通常取相邻面积的中点：

$$55,\ 70,\ 90,\ 110$$

比如阈值 $70$ 表示：

$$面积\le70$$

左边是面积 50、60 的房子；右边是面积 80、100、120 的房子。

## 7. 手算阈值 70 的 SSE

阈值 $70$ 的划分结果：

| 分支 | 房价 |
| --- | --- |
| 左：面积 $\le70$ | 100, 120 |
| 右：面积 $>70$ | 160, 200, 240 |

先算左分支均值：

$$\bar{y}_{left}=\frac{100+120}{2}=110$$

左分支平方误差：

$$SSE_{left}=(100-110)^2+(120-110)^2$$

$$SSE_{left}=100+100=200$$

再算右分支均值：

$$\bar{y}_{right}=\frac{160+200+240}{3}=200$$

右分支平方误差：

$$SSE_{right}=(160-200)^2+(200-200)^2+(240-200)^2$$

$$SSE_{right}=1600+0+1600=3200$$

所以阈值 $70$ 的划分后总误差是：

$$SSE_{split}=200+3200=3400$$

## 8. 比较所有候选阈值

我们把所有候选阈值都算一遍：

| 阈值 | 左分支房价 | 右分支房价 | 划分后 SSE |
| ---: | --- | --- | ---: |
| 55 | 100 | 120, 160, 200, 240 | 8000 |
| 70 | 100, 120 | 160, 200, 240 | 3400 |
| 90 | 100, 120, 160 | 200, 240 | 3400 |
| 110 | 100, 120, 160, 200 | 240 | 8000 |

最小的划分后 SSE 是：

$$3400$$

所以阈值 $70$ 和 $90$ 都可以作为当前最优划分。

如果模型遇到并列最优，具体选择哪个阈值可能和实现细节有关。

In [ ]:
import pandas as pd

data = pd.DataFrame({
    '面积': [50, 60, 80, 100, 120],
    '房价': [100, 120, 160, 200, 240],
})

data

In [ ]:
def sse(values):
    """计算一组数值相对于自身均值的平方误差。"""
    mean_value = values.mean()
    return ((values - mean_value) ** 2).sum()


areas = data['面积'].tolist()
thresholds = [(areas[i] + areas[i + 1]) / 2 for i in range(len(areas) - 1)]

results = []

for threshold in thresholds:
    left = data[data['面积'] <= threshold]
    right = data[data['面积'] > threshold]
    split_sse = sse(left['房价']) + sse(right['房价'])
    results.append({
        '阈值': threshold,
        '左分支房价': left['房价'].tolist(),
        '右分支房价': right['房价'].tolist(),
        '划分后SSE': split_sse,
    })

pd.DataFrame(results)

## 9. 用 sklearn 训练 CART 回归树

在 sklearn 中，回归树模型是：

```python
DecisionTreeRegressor
```

分类树是：

```python
DecisionTreeClassifier
```

两者名字很像，但目标不同：

- `Classifier`：预测类别。
- `Regressor`：预测连续值。

In [ ]:
from sklearn.tree import DecisionTreeRegressor

X = data[['面积']]
y = data['房价']

model = DecisionTreeRegressor(max_depth=1, random_state=22)
model.fit(X, y)

print('树深度：', model.get_depth())
print('叶子节点数量：', model.get_n_leaves())

In [ ]:
test_area = pd.DataFrame({'面积': [55, 75, 110]})
predictions = model.predict(test_area)

pd.DataFrame({
    '面积': test_area['面积'],
    '预测房价': predictions,
})

## 10. 回归树为什么是分段常数模型？

回归树的每个叶子节点输出一个固定均值。

所以它的预测结果不是一条平滑曲线，而是一段一段的常数。

例如：

```text
面积 <= 70      -> 预测 110
面积 > 70       -> 预测 200
```

如果树更深，它会继续切分，预测会变成更多小台阶。

这也是回归树的特点：

$$\text{用一段一段的均值去近似连续关系。}$$

## 11. CART 分类树和回归树总结对比

| 对比点 | CART 分类树 | CART 回归树 |
| --- | --- | --- |
| 预测目标 | 类别 | 连续值 |
| 叶子节点输出 | 多数类别 | 标签均值 |
| 划分指标 | Gini 指数 | 平方误差 SSE / MSE |
| 目标 | 子节点类别更纯 | 子节点数值更集中 |
| sklearn 模型 | `DecisionTreeClassifier` | `DecisionTreeRegressor` |

一句话记忆：

$$\text{分类树让类别更纯，回归树让数值更集中。}$$

## 12. 本课小结

- CART 不只有分类树，也有回归树。
- 回归树用于预测连续值。
- 回归树的叶子节点通常输出该叶子样本标签的平均值。
- 回归树用平方误差衡量节点内数值是否集中。
- 划分时选择使左右子节点总平方误差最小的切法。
- sklearn 中使用 `DecisionTreeRegressor` 构建回归树。

记忆方式：

$$\text{CART 回归树 = 二叉划分 + 叶子均值 + 最小平方误差}$$